In [3]:
import pandas as pd
from pathlib import Path

folder = Path.home() / "Desktop" / "Fall 2026 Classes"

for file in folder.rglob("*"):
    if "HiSeqV2" in file.name:
        print(file)


/Users/skylerskinner/Desktop/Fall 2026 Classes/Datasets for Checkpoint 1/TCGA.LUAD.sampleMap_HiSeqV2.gz


In [5]:
files = list(folder.rglob("*HiSeqV2*"))

file_path = files[0]

expression = pd.read_csv(
    file_path,
    sep="\t",
    compression="infer",
    index_col=0
)

print("Dataset shape:", expression.shape)
expression.iloc[:5, :5]

Dataset shape: (20530, 576)


,TCGA-69-7978-01,TCGA-62-8399-01,TCGA-78-7539-01,TCGA-50-5931-11,TCGA-73-4658-01
sample,,,,,
ARHGEF10L,9.9898,10.4257,9.6264,8.6835,9.2078
HIF3A,4.2598,11.6239,9.1362,9.4824,5.0288
RNF17,0.4181,0.0000,1.1231,0.8221,0.0000
RNF10,10.3657,11.5489,11.6692,11.7341,11.6209
RNF11,11.1718,11.0200,10.4679,11.6787,11.3414


In [6]:
#gene expression features
expression_features = pd.DataFrame({
    "mean_expression": expression.mean(axis=1),
    "expression_variability": expression.std(axis=1)
})

print("Feature table shape:", expression_features.shape)
expression_features.head()

Feature table shape: (20530, 2)


,mean_expression,expression_variability
sample,,
ARHGEF10L,9.538420,0.765509
HIF3A,7.109322,2.222825
RNF17,0.546533,0.938638
RNF10,11.766623,0.344360
RNF11,10.959373,0.492240


In [7]:
output_path = folder / "Datasets for Checkpoint 1" / "LUAD_expression_features.csv"

expression_features.to_csv(output_path)

print("Saved to:", output_path)

Saved to: /Users/skylerskinner/Desktop/Fall 2026 Classes/Datasets for Checkpoint 1/LUAD_expression_features.csv


In [9]:
from pathlib import Path

folder = Path.home() / "Desktop" / "Fall 2026 Classes"

data_folder = folder / "Datasets for Checkpoint 1"

for file in data_folder.rglob("*"):
    if file.is_file() and (
        "cna" in file.name.lower()
        or "gistic" in file.name.lower()
    ):
        print(file)

In [10]:
for file in data_folder.rglob("*"):
    if file.is_file():
        print(file.name)

luad_tcga.tar.gz
CRISPRGeneEffect.csv
LUAD_expression_features.csv
subcellular_location.tsv
TCGA.LUAD.sampleMap_HiSeqV2.gz


In [12]:
import tarfile

archive_path = data_folder / "luad_tcga.tar.gz"

with tarfile.open(archive_path, "r:gz") as tar:
    tar.extractall(path=data_folder, filter="data")

print("Extraction complete!")

Extraction complete!


In [13]:
for file in data_folder.rglob("*"):
    if file.is_file() and (
        "cna" in file.name.lower()
        or "gistic" in file.name.lower()
    ):
        print(file)

/Users/skylerskinner/Desktop/Fall 2026 Classes/Datasets for Checkpoint 1/luad_tcga/meta_cna.txt
/Users/skylerskinner/Desktop/Fall 2026 Classes/Datasets for Checkpoint 1/luad_tcga/data_linear_cna.txt
/Users/skylerskinner/Desktop/Fall 2026 Classes/Datasets for Checkpoint 1/luad_tcga/data_cna.txt
/Users/skylerskinner/Desktop/Fall 2026 Classes/Datasets for Checkpoint 1/luad_tcga/meta_gistic_genes_del.txt
/Users/skylerskinner/Desktop/Fall 2026 Classes/Datasets for Checkpoint 1/luad_tcga/data_cna_hg19.seg
/Users/skylerskinner/Desktop/Fall 2026 Classes/Datasets for Checkpoint 1/luad_tcga/meta_gistic_genes_amp.txt
/Users/skylerskinner/Desktop/Fall 2026 Classes/Datasets for Checkpoint 1/luad_tcga/data_gistic_genes_del.txt
/Users/skylerskinner/Desktop/Fall 2026 Classes/Datasets for Checkpoint 1/luad_tcga/meta_cna_hg19_seg.txt
/Users/skylerskinner/Desktop/Fall 2026 Classes/Datasets for Checkpoint 1/luad_tcga/data_gistic_genes_amp.txt
/Users/skylerskinner/Desktop/Fall 2026 Classes/Datasets for Che

In [15]:
cna_path = data_folder / "luad_tcga" / "data_cna.txt"
cna = pd.read_csv(cna_path, sep="\t")

print("Dataset shape:", cna.shape)
print("Columns:", cna.columns[:8].tolist())

cna.iloc[:5, :8]

Dataset shape: (24776, 518)
Columns: ['Hugo_Symbol', 'Entrez_Gene_Id', 'TCGA-05-4244-01', 'TCGA-05-4249-01', 'TCGA-05-4250-01', 'TCGA-05-4382-01', 'TCGA-05-4384-01', 'TCGA-05-4389-01']


,Hugo_Symbol,Entrez_Gene_Id,TCGA-05-4244-01,TCGA-05-4249-01,TCGA-05-4250-01,TCGA-05-4382-01,TCGA-05-4384-01,TCGA-05-4389-01
0,ACAP3,116983.0,-1,-1,-1,0,0,-1
1,ACTRT2,140625.0,-1,-1,-1,0,0,-1
2,AGRN,375790.0,-1,-1,-1,0,0,-1
3,ANKRD65,441869.0,-1,-1,-1,0,0,-1
4,ATAD3A,55210.0,-1,-1,-1,0,0,-1


In [16]:
sample_columns = cna.columns[2:]

cna_values = cna[sample_columns].apply(
    pd.to_numeric, errors="coerce"
)

amplication_features = pd.DataFrame({
    "Gene": cna["Hugo_Symbol"],
    "amplication_frequency": (
        cna_values.eq(2).sum(axis=1)
        / cna_values.notna().sum(axis=1)
    )
})

amplification_features = amplication_features.set_index("Gene")
print("Amplification Features Shape:", amplification_features.shape)
amplification_features.head()

Amplification Features Shape: (24776, 1)


,amplication_frequency
Gene,
ACAP3,0.017442
ACTRT2,0.017442
AGRN,0.017442
ANKRD65,0.017442
ATAD3A,0.017442


In [17]:
#check for duplicate gene names
print("Expression duplicates:", expression_features.index.duplicated().sum())
print("Amplification duplicates:", amplification_features.index.duplicated().sum())

Expression duplicates: 0
Amplification duplicates: 11


In [18]:
#resolve amplification duplicates
duplicates = amplification_features[
    amplification_features.index.duplicated(keep=False)
]

print(duplicates.sort_index())

            amplication_frequency
Gene                             
ABHD16B                  0.060078
ABHD16B                  0.060078
C1orf220                 0.062016
C1orf220                 0.062016
COMMD9                   0.013566
COMMD9                   0.013566
FOXD4L3                  0.005814
FOXD4L3                  0.005814
GADL1                    0.007752
GADL1                    0.000000
PALM2AKAP2               0.003876
PALM2AKAP2               0.003876
PDPK1                    0.005814
PDPK1                    0.005814
SLCO1B3                  0.042636
SLCO1B3                  0.042636
TIMP2                    0.031008
TIMP2                    0.031008
TRAPPC2                  0.005814
TRAPPC2                  0.003876
XAGE1A                   0.007752
XAGE1A                   0.007752


In [ ]:
#duplicate genes with different amplification frequencies
conflicting_genes = amplification_features.groupby(level=0)[
    "amplification_frequency"
].nunique()

conflicting_genes = conflicting_genes[conflicting_genes > 1].index

print("Conflicting genes:", conflicting_genes.tolist())

#remove conflicting genes
amplification_clean = amplification_features[
    ~